# 04 · Orbitals, interference, phase transitions and phase space

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/notebooks/04_orbitals_interference_and_phase_space.ipynb)

New modules in this notebook: `quantum` (orbitals, 2D dynamics), `statmech`, `hamiltonian`, `dynamics`, `complexplane` and `linalg`.

In [ ]:
# Setup: installs the qplay package if it is missing (for example on Google Colab, about 20 s).
import importlib.util, subprocess, sys
if importlib.util.find_spec("qplay") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/StativeSh/quantum-playground"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from qplay import quantum, statmech, hamiltonian, dynamics, complexplane, linalg, integrators

## Hydrogen orbitals

$\psi_{nlm}(r,\theta,\phi) = R_{nl}(r)\,Y_l^m(\theta,\phi)$. The radial part has $n-l-1$ nodes, and $\langle r\rangle = \tfrac{a_0}{2}\left[3n^2 - l(l+1)\right]$.

In [ ]:
r = np.linspace(0, 30, 600)
for n, l in [(1, 0), (2, 0), (2, 1), (3, 0), (3, 1), (3, 2)]:
    plt.plot(r, r**2 * quantum.hydrogen_radial(n, l, r) ** 2, label=f"n={n}, l={l}")
plt.xlabel(r"$r/a_0$"); plt.ylabel(r"$r^2 R_{nl}^2$"); plt.legend(); plt.title("Radial probability density");

In [ ]:
X, Z, rho = quantum.hydrogen_density_xz(3, 2, 0)
plt.imshow(rho ** 0.5, origin="lower", extent=[X.min(), X.max(), Z.min(), Z.max()], cmap="inferno")
plt.title(r"$3d_{z^2}$ orbital"); plt.grid(False);

## The double-slit experiment

A single wavepacket passes through two slits. Dark fringes appear where the path difference from the two slits is $(m+\tfrac12)\lambda$, with $\lambda = 2\pi/k_0$.

In [ ]:
run = quantum.double_slit_experiment()
y, L, lam, d = run["y"], run["screen_x"], 2 * np.pi / 3.0, 5.0
path_diff = np.hypot(L, y + d / 2) - np.hypot(L, y - d / 2)
plt.plot(y, run["screen"], label="simulation")
plt.plot(y, np.cos(np.pi * path_diff / lam) ** 2, "k:", label="two point sources")
plt.xlim(-30, 30); plt.xlabel("y on screen"); plt.legend();

## The Ising model and a phase transition

Below Onsager's $T_c = 2/\ln(1+\sqrt2)\approx 2.269$ the spins order spontaneously, with $m = \left(1-\sinh^{-4}(2/T)\right)^{1/8}$.

In [ ]:
temps = np.linspace(1.4, 3.4, 15)
mags = [np.mean(np.abs(statmech.ising(L=32, T=T, sweeps=400, seed=1)[1][150:])) for T in temps]
fine = np.linspace(1.0, 3.4, 300)
plt.plot(fine, statmech.onsager_magnetization(fine), "k", label="exact")
plt.plot(temps, mags, "o", label="Monte Carlo")
plt.axvline(statmech.T_CRITICAL, color="r", lw=0.5); plt.xlabel("T"); plt.ylabel(r"$\langle|m|\rangle$"); plt.legend();

## From Lagrangian to Hamiltonian

The Legendre transform $p = \partial L/\partial\dot q$, $H = p\dot q - L$ gives the pendulum Hamiltonian; Hamilton's equations then drive the simulation.

In [ ]:
t, m, l, g = sp.symbols("t m l g", positive=True)
th = sp.Function("theta")(t)
L = m * l**2 * sp.diff(th, t)**2 / 2 + m * g * l * sp.cos(th)
H, (q,), (p,) = hamiltonian.legendre_transform(L, [th], t)
H

In [ ]:
params = {m: 1, l: 1, g: 1}
rhs = hamiltonian.to_numeric_rhs(H, [q], [p], params)
for q0 in (0.5, 1.5, 2.5, 3.0):
    _, y = integrators.solve(rhs, [q0, 0.0], (0, 20), n=800)
    plt.plot(y[:, 0], y[:, 1], lw=1)
plt.xlabel("q"); plt.ylabel("p"); plt.title("Pendulum orbits in phase space");

## Phase planes and fixed points

Each fixed point is classified from the trace $\tau$ and determinant $\Delta$ of the Jacobian.

In [ ]:
def van_der_pol(s, mu=1.0):
    return [s[1], mu * (1 - s[0] ** 2) * s[1] - s[0]]

dynamics.phase_portrait(van_der_pol, (-3, 3), (-4, 4), classify=[(0.1, 0.1)])
plt.title("Van der Pol oscillator: an unstable spiral feeding a limit cycle");

## Complex functions

Domain colouring shows $\arg f$ as hue. Around each zero the colours cycle once per order, and they run the other way around a pole. The argument principle counts zeros minus poles from the winding number.

In [ ]:
f = lambda z: (z**3 - 1) / (z**2 + 0.5)
plt.imshow(complexplane.domain_coloring(f, (-2, 2), (-2, 2), 600), origin="lower", extent=[-2, 2, -2, 2])
plt.title(f"zeros - poles inside |z|=1.5: {complexplane.winding_number(f, 0, 1.5)}"); plt.grid(False);

## Linear maps

A matrix sends the unit circle to an ellipse whose semi-axes are the singular values times the left singular vectors.

In [ ]:
A = linalg.shear(0.8) @ np.diag([1.5, 0.6])
sigma, U = linalg.singular_axes(A)
plt.plot(*linalg.unit_circle_image(np.eye(2)), color="0.6")
plt.plot(*linalg.unit_circle_image(A))
for i in range(2):
    plt.arrow(0, 0, *(sigma[i] * U[:, i]), width=0.02, color="C1", length_includes_head=True)
plt.gca().set_aspect("equal"); plt.title(f"singular values {np.round(sigma, 3)}");

## Your turn

- Plot $|\psi|^2$ for the 4f orbitals with $m=0,1,2,3$. How does the number of nodal cones depend on $l-|m|$?
- Narrow the slits in `double_slit_experiment` and watch the single-slit envelope shrink the side fringes.
- Add a field `h` to the Ising model and watch the symmetry break.